### Extraer df datos_colas completo

In [23]:
from db_conn import conn_db_wf
import polars as pl
engine, metadata = conn_db_wf()
conn = engine.connect()
query = """
        SELECT rendimiento_id, fecha, intervalo, tipo_medios, cola_id, id_cola, nombre_cola, nombre_cola_dimension, oferta, contestadas, abandonadas, saliente, flujo_saliente, velocidad_media_respuesta_seg, porcentaje_nivel_servicio, porcentaje_objetivo_nivel_servicio, espera_media_seg, manejo_medio_seg, conversacion_media_seg, retencion_media_seg, acw_medio_seg, abandono_medio_seg, manejo, retener, transferir, tiempo_manejo_total, cumplen_sla, tiempo_conversacion_total, tiempo_retencion_total, tiempo_acw_total, abandono_rapido, salientes_intentadas, campana_id, nombre_campana, tmo_objetivo, nivel_atencion_objetivo, nivel_servicio_objetivo, tiempo_maximo_espera_objetivo, observaciones_campana, cliente_id, horario_id, plataforma_operacion_id, jefe_negocio_id, unidad_negocio_id, grupo_facturacion_id, tipo_campana_id, tipo_campana, cliente, plataforma_operacion, unidad_negocio, jefe_negocio, supervisores
        FROM reporting.datos_colas
"""
df = pl.read_database(query, conn, infer_schema_length=None, batch_size=10000)
# df.head()

columnas_time = [
    nombre
    for nombre, dtype in df.schema.items()
    if dtype == pl.Time
]

df_excel = df.with_columns(
    [
        pl.col(col).dt.strftime("%H:%M:%S").alias(col)
        for col in columnas_time
    ]
)

In [24]:
from collections import defaultdict

tipos = defaultdict(list)

for columna, dtype in df.schema.items():
    tipos[str(dtype)].append(columna)

for dtype, columnas in tipos.items():
    print(f"\n{dtype}")
    for columna in columnas:
        print(f"  - {columna}")


Int64
  - rendimiento_id
  - cola_id
  - oferta
  - contestadas
  - abandonadas
  - saliente
  - flujo_saliente
  - manejo
  - retener
  - transferir
  - cumplen_sla
  - abandono_rapido
  - salientes_intentadas
  - campana_id
  - cliente_id
  - horario_id
  - plataforma_operacion_id
  - jefe_negocio_id
  - unidad_negocio_id

Date
  - fecha

Time
  - intervalo

String
  - tipo_medios
  - id_cola
  - nombre_cola
  - nombre_cola_dimension
  - nombre_campana
  - observaciones_campana
  - cliente
  - plataforma_operacion
  - unidad_negocio
  - jefe_negocio
  - supervisores

Decimal(precision=38, scale=6)
  - velocidad_media_respuesta_seg
  - espera_media_seg
  - manejo_medio_seg
  - conversacion_media_seg
  - retencion_media_seg
  - acw_medio_seg
  - abandono_medio_seg

Float64
  - porcentaje_nivel_servicio
  - porcentaje_objetivo_nivel_servicio

Decimal(precision=38, scale=4)
  - tiempo_manejo_total
  - tiempo_conversacion_total
  - tiempo_retencion_total
  - tiempo_acw_total

Null
  - tm

### Pruebas base xlwings

In [11]:
import xlwings as xw

app = xw.App(visible=True, add_book=True)
wb = app.books[0]
hoja = wb.sheets[0]
hoja["A1"].value = "Prueba 1"
hoja["B1"].value = 123
wb.save(r".\archivos\prueba.xlsx")
wb.close()
app.quit()


#### Prueba con with

In [ ]:
import xlwings as xw

with xw.App(visible=False, add_book=True) as app:
    wb = app.books[0]
    hoja = wb.sheets[0]
    hoja["A1"].value = "Prueba 1"
    hoja["B1"].value = 123
    wb.save(r".\archivos\prueba.xlsx")

In [22]:
import xlwings as xw

ruta = r".\archivos\reporte.xlsx"
tamano_bloque = 10_000

with xw.App(visible=False, add_book=False) as app:
    # with app.properties(
    #     calculation="manual",
    #     screen_updating=False,
    #     display_alerts=False
    # ):
    app.screen_updating = False
    app.display_alerts = False
    # app.calculation = "manual"

    libro = app.books.add()

    hoja = libro.sheets[0]
    hoja.name = "Datos"

    # Encabezados
    hoja.range("A1").value = [df.columns]

    # Datos por bloques
    for inicio in range(0, df.height, tamano_bloque):
        # bloque = df.slice(inicio, tamano_bloque)
        bloque = df_excel.slice(inicio, tamano_bloque)

        datos = bloque.rows()

        fila_excel = inicio + 2

        hoja.range((fila_excel, 1)).value = datos

    # Rango completo
    rango_tabla = hoja.range(
        (1, 1),
        (df.height + 1, df.width)
    )

    # Crear tabla
    tabla = hoja.tables.add(
        source=rango_tabla,
        name="t_datos",
        has_headers=True,
        table_style_name="TableStyleMedium9"
    )

    tabla.show_table_style_first_column = False
    tabla.show_table_style_last_column = False
    tabla.show_table_style_row_stripes = True
    tabla.show_table_style_column_stripes = True

    libro.save(ruta)

TypeError: Objects of type 'datetime.time' can not be converted to a COM VARIANT

In [5]:
campanas = pl.read_database(
    """
    SELECT DISTINCT campana_id, nombre_campana
    FROM reporting.datos_colas
    WHERE campana_id IS NOT NULL
    """,
    engine
)

In [6]:
print(campanas)

shape: (26, 2)
┌────────────┬─────────────────────────────────┐
│ campana_id ┆ nombre_campana                  │
│ ---        ┆ ---                             │
│ i64        ┆ str                             │
╞════════════╪═════════════════════════════════╡
│ 1          ┆ RS IN Examenes                  │
│ 2          ┆ Mutual - Consultas Generales    │
│ 3          ┆ Integramédica Mesa de Soporte … │
│ 4          ┆ Colonial habil                  │
│ 5          ┆ Cruz Verde                      │
│ …          ┆ …                               │
│ 62         ┆ Out Ges Colmena                 │
│ 68         ┆ WSA                             │
│ 69         ┆ Clinica Pasteur                 │
│ 70         ┆ Mutual - Agenda Asbesto         │
│ 71         ┆ Bupa Conecta                    │
└────────────┴─────────────────────────────────┘


In [ ]:
for campana in campanas.iter_rows(named=True):

    query = """
        SELECT ...
        FROM reporting.datos_colas
        WHERE campana_id = :campana_id
    """

    df = pl.read_database(
        query,
        engine,
        execute_options={
            "parameters": {
                "campana_id": campana["campana_id"]
            }
        }
    )

    # copiar plantilla
    # abrir con xlwings